In [75]:
import pandas as pd
import numpy as np
import calendar
from pathlib import Path

# Cleaning the raw data we get from NHS A&E attendance

In [76]:
# Generate list of month names (January to December)
months = list(calendar.month_name)[1:]

# Define year range (2018–2025 inclusive)
years = list(range(2018, 2026))

# Read Excel file:
# - skip first 14 rows (metadata/headers not needed)
# - header is multi-level (2 rows)
# - select specific sheet
def process_file(year, month):  
    try:
        df_prov = pd.read_excel(
            f'./Datasets/Raw/{year}/{month}.xls',
            skiprows=14,
            header=[0, 1],
            sheet_name='Provider Level Data'
        )
    except Exception as e:
        print(f"Error reading {year}-{month}: {e}")

    # Remove first and last columns (usually empty or totals not needed)
    df_prov = df_prov.iloc[:, 1:-1]


    # Flatten multi-level column headers:
    # - remove 'Unnamed' columns (artifacts from Excel merged cells)
    # - clean spaces and unwanted characters
    # - join levels using '_'
    df_prov.columns = [
        '_'.join([
            str(lvl).strip()
            .replace('Unnamed:', '')
            .replace(' - ', '_')
            .strip('_ ')
            for lvl in col
            if pd.notna(lvl) and 'Unnamed' not in str(lvl)
        ])
        for col in df_prov.columns
    ]


    # Drop columns containing percentage values (not required for analysis)
    df_prov = df_prov.loc[:, ~df_prov.columns.str.contains('Percentage')]


    # Add metadata columns for tracking
    df_prov['Month'] = month
    df_prov['Year'] = year


    # Rename columns to a consistent schema:
    # This ensures all files align structurally for downstream analysis
    new_cols = [
        'Code', 'Region', 'Name',
        'Attend_Type1', 'Attend_Type2', 'Attend_Type3', 'Attend_Total',
        'Within4Hr_Type1', 'Within4Hr_Type2', 'Within4Hr_Type3', 'Within4Hr_Total',
        'Over4Hr_Type1', 'Over4Hr_Type2', 'Over4Hr_Type3', 'Over4Hr_Total',
        'Adm_AE_Type1', 'Adm_AE_Type2', 'Adm_AE_Type3', 'Adm_AE_Total',
        'Adm_Other', 'Adm_Total',
        'DecisionToAdm_Wait_4to12Hr', 'DecisionToAdm_Wait_Over12Hr',
        'Month', 'Year'
    ]

    if len(df_prov.columns) != len(new_cols):
        print(f"Column mismatch in {year}-{month}")
        return None
    
    df_prov.columns = new_cols


    # Remove rows with any missing values (missing code data)
    # (ensures clean dataset for analysis)
    df_prov = df_prov.dropna(subset=['Code'])


    # Remove invalid rows where provider code is '-'
    # (these are usually summary or placeholder rows)
    df_prov = df_prov[df_prov['Code'] != '-'].reset_index(drop=True)

    return df_prov


## Merging dataframes and creating and exporting final dataset

In [77]:
### creating 1 final file
all_data = []

for year in years:
    for month in months:
        df = process_file(year, month)
        if df is not None:
            all_data.append(df)

# Combine all into one DataFrame
final_df = pd.concat(all_data, ignore_index=True)

In [78]:
#exporting final df into csv
final_df.to_csv('./Datasets/Cleaned/master_dataset.csv', index=False)